In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from datetime import datetime

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Meiryo', 'Yu Gothic', 'Hiragino Sans', 'AppleGothic']



LEVELS = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])


def to_index(values):
    return np.array([np.argmin(np.abs(LEVELS - v)) for v in values])

def calc_score(y_true, y_pred):
    from sklearn.metrics import r2_score, mean_absolute_error
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return float(r2), float(mae)

def draw_confusion_matrix(y_true, y_pred, title, r2, mae, save_path):
    y_true_idx = to_index(y_true)
    y_pred_idx = to_index(y_pred)

    # view_result()と同じ
    cm = confusion_matrix(y_true_idx, y_pred_idx).T
    cm = np.flipud(cm)

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=LEVELS,
        yticklabels=LEVELS[::-1],
    )

    plt.title(f"{title}\nN={len(y_true)}, R²={r2:.3f}, MAE={mae:.3f}", fontsize=15)
    # plt.title(title.format(mae), fontsize=20)
    plt.xlabel("実測値", fontsize=13)
    plt.ylabel("システム推定値", fontsize=13)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved : {save_path}")


# ------------------------------------------------------------
# 設定読み込み
# ------------------------------------------------------------

with open("final_results.json", encoding="utf-8") as f:
    results = json.load(f)

plate_settings = {}

# plate_setting_A.json ～ plate_setting_D.json
for plate_id in ["A", "B", "C", "D"]:
    with open(f"test_data/{plate_id}/plate_setting.json", encoding="utf-8") as f:
        setting = json.load(f)

    plate_settings[plate_id] = {
        item["plate_name"]: item["nansai"]
        for item in setting
    }


# ------------------------------------------------------------
# データ集約
# ------------------------------------------------------------

true_nansai = []
pred_nansai = []

true_other = []
pred_other = []

for result in results:

    plate_id = result["plate_id"]
    plate_name = result["plate_name"]
    
    # if plate_id=="A" and plate_name in ["fukusai2","fukusai3"]:
    #     continue

    nansai = plate_settings[plate_id][plate_name]

    if nansai:
        true_nansai.extend(result["y_true"])
        pred_nansai.extend(result["y_pred"])
    else:
        true_other.extend(result["y_true"])
        pred_other.extend(result["y_pred"])


r2_nansai, mae_nansai = calc_score(true_nansai, pred_nansai)
r2_other, mae_other = calc_score(true_other, pred_other)



os.makedirs("result_analyze", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

draw_confusion_matrix(
    1 - np.array(true_nansai),
    1- np.array(pred_nansai),
    # "Confusion Matrix: Liquid Food Intake Results",
    "液体食品推定結果",
    r2_nansai,
    mae_nansai,
    f"result_analyze/cm_nansai_{timestamp}.png",
)

draw_confusion_matrix(
    1 - np.array(true_other),
    1 - np.array(pred_other),
    "固体食品推定結果",
    r2_other,
    mae_other,
    f"result_analyze/cm_not_nansai_{timestamp}.png",
)

In [ ]:
import cv2

img = cv2.imread("2026-07-21_210343.png", cv2.IMREAD_GRAYSCALE)
cv2.imwrite("output.png", img)